# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 9


## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un program de calculator extrem de mare, antrenat pe cantități masive de text, capabil să înțeleagă, să genereze și să manipuleze limbajul uman într-un mod coerent și relevant. Acest lucru îi permite să efectueze o gamă largă de sarcini lingvistice, precum răspunsuri la întrebări, scrierea de texte creative, traduceri și rezumate.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenat pe cantități masive de text pentru a înțelege și genera limbaj uman. Aceste modele pot realiza diverse sarcini lingvistice, cum ar fi răspunsuri la întrebări, traduceri, rezumate și chiar crearea de conținut nou.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, de ce e mai bine să pui laptele primul în locul cerealelor.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
Puneți laptele primul pentru a evita stropirea și a asigura o amestecare uniformă a cerealelor. Astfel, veți obține o consistență perfectă, fără cocoloașe sau lapte rămas pe fundul bolului.

--- Gemini 2.5 Flash ---
Punerea laptelui primul permite un control mai bun asupra cantității, asigurând că nu se toarnă prea mult și că cerealele nu se înmoaie excesiv de repede. Astfel, cerealele rămân crocante mai mult timp, oferind o textură optimă la fiecare lingură.

--- OpenRouter Free ---
Pentru a pune la locul cerealelor, e necesită o soluție clară. Dacă vorbi despre locații de mici, pot spune că locații simplici sunt ideale. Acesta este 2 propoziții scurte și concise.


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [7]:
SYSTEM = """
Ești un pirat care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Apa este informație, iar noi suntem scufundați în ea. Unii înoată, alții se îneacă, iar politicienii profită de pe urma valurilor."

Răspunde în 4 linii:
Ton: negativ
Emoție dominantă: dispreț
Țintă principală: politicieni
Populism: da
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Tonul este negativ, plin de dispreț față de politicieni.
Aceștia sunt văzuți ca profitând de situația generală, sugerând o lipsă de scrupule.
Comentariul folosește o metaforă pentru a critica inegalitatea și exploatarea, elemente populiste.
Se sugerează că politicienii se folosesc de "valuri" (situații) pentru propriul câștig.

--- Gemini 2.5 Flash ---
Ton: Negativ, ca o furtună pe mare.
Emoție dominantă: Dispreț adânc, ca pentru un șobolan de navă.
Țintă principală: Politicienii, niște pirați fără onoare.
Populism: Da, un discurs ce-i aruncă pe toți în aceeași apă.

--- OpenRouter Free ---
Metaforă care identifică politicienii ca profitori de pe urma crizei informaționale, cultivând resentimentul populist prin divizarea "noi" vs. "ei". Disprețuos.


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [8]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [10]:
COMENTARIU = "Au castigat Reform in Anglia la alegerile locale, ceea ce arata ca oamenii vor schimbare si sunt satui de partidele traditionale. O sa se aleaga praful de popor."

SYSTEM = "Ești un politician de la Greens care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'dezamagire', 'tinta_principala': 'partidele traditionale', 'populism': False, 'explicatie_scurta': 'Comentariul exprimă dezamăgire față de rezultatele alegerilor locale, sugerând că victoria unui partid nou (Reform) indică o dorință de schimbare din partea electoratului, dar anticipează un deznodământ negativ pentru populație.'}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'dezamagire', 'tinta_principala': 'Alegatorii si partidele traditionale', 'populism': True, 'explicatie_scurta': 'Comentariul reflectă o retorică populistă, speculând pe nemulțumirea față de partidele tradiționale, dar exprimă o viziune fatalistă asupra viitorului, fără a oferi soluții constructive. Schimbarea este necesară, dar nu orice schimbare este un progres.'}

--- OpenRouter Free ---


KeyboardInterrupt: 

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [11]:
PROMPT_STAB = """
Ne aflam in apocalipsa insectelor.
Explică în 2 propoziții ce poate însemna acest lucru pentru lanturile trofice si pentru umanitate.
Răspunde neutru, structurat si ferm pentru actiune.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Apocalipsa insectelor ar perturba drastic lanțurile trofice, ducând la colapsul populațiilor de animale dependente de insecte pentru hrană și la o scădere semnificativă a polenizării, afectând producția agricolă. Pentru umanitate, aceasta ar însemna o criză alimentară severă, pierderea biodiversității și un impact negativ profund asupra ecosistemelor globale.

temperature=0.7:
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

temperature=1.2:
Colapsul populațiilor de insecte ar destabiliza lanțurile trofice prin eliminarea unor producători și consumatori esențiali, afectând direct organismele de la niveluri superioare și implicit ecosistemele. Pentru umanitate, aceasta ar însemna o amenințare majoră la adresa securității alimentare, a serviciilor ecosistemice vitale (polenizare, descompunere) și a biodiversității, impunând măsuri de conservare și restaurare urgent

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da / nu / **parțial** | da / nu / **parțial** | da / nu / **parțial** | **da** / nu | Se vede ca este Lite. |
| Gemini 2.5 Flash | **da** / nu / parțial | **da** / nu / parțial | **da** / nu / parțial | da / **nu** | Mai mereu raspunsul asta era mai corect si concret decat counterpart-ul Lite. |
| OpenRouter Free | da / **nu** / parțial | da / **nu** / parțial | Nici nu a rulat dupa 10 minute. | **da** / nu | E dus pe ulei OpenRouter. |
### Decizie
**Model principal ales: **Gemini 2.5 Flash**  
**Model de rezervă: **Gemini 2.5 Flash Lite**  
**Temperature recomandată: **0.1**  
**De ce am ales acest model?**  
Mdaa, se poate folosi Gemini pe baza modelelor incercate. Care dintre cele doua, acum este o dilema de tokens, credite si daca merita acel raspuns un pic mai detaliat costurile aditionale. Dupa mine am putea merge pentru Gemini 2.5 Flash pentru performanta putin mai buna, OpenRouter e dus cu pluta.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [17]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales